In [1]:
import tensorboard
tensorboard.__version__

'2.20.0'

### Phase 1: Model dự đoán kí hiệu đơn lẻ

In [ ]:
import tensorflow as tf

def build_isolated_sign_model(
        num_keypoints=128,  # 21 keypoints x 2 tay
        feature_dim=256,
        num_heads=8,
        num_classes=30,
        dropout_rate=0.3,
        max_frames=180
):
    inputs = tf.keras.Input(shape=(max_frames, num_keypoints), name="keypoints_input")

    # Positional Encoding
    pos_encoding = tf.keras.layers.Embedding(input_dim=max_frames, output_dim=feature_dim)(
        tf.range(start=0, limit=max_frames))
    x = tf.keras.layers.Dense(feature_dim)(inputs)
    x = x + pos_encoding

    # Transformer Encoder
    for _ in range(3):
        attn_output = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=feature_dim // num_heads)(x, x)
        attn_output = tf.keras.layers.Dropout(dropout_rate)(attn_output)
        x = tf.keras.layers.LayerNormalization()(x + attn_output)

        ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(feature_dim * 4, activation="relu"),
            tf.keras.layers.Dense(feature_dim),
            tf.keras.layers.Dropout(dropout_rate)
        ])
        x = tf.keras.layers.LayerNormalization()(x + ffn(x))

    # Pooling và phân loại
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name="IsolatedSignModel")
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

if __name__ == "__main__":
    model = build_isolated_sign_model()
    model.summary()


In [34]:
import numpy as np 

a = np.load(r"C:\Users\ming2\Documents\FPT_University\Semester 5\DPL302m\Project\Sign_Language_Translation\Data_Keypoints\test\Accept\6.npy")
print(a.shape)

print(a[:, -2:])

(61, 128)
[[1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]]


Sắp xếp chuỗi keypoint theo từng khung hình và theo thứ tự từ tay trái tới tay phải.
